[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/InterstellarNerd8760/lunar-atlas/blob/proxima/terrain-notebooks/notebooks/prepare-sldem-band.ipynb)

# SLDEM2015: sharper terrain for the middle of the Moon (60°S to 60°N)

SLDEM2015 combines LOLA laser altimetry with Kaguya (SELENE) Terrain Camera stereo, so it fills the gaps between LOLA tracks near the equator. It only covers 60°S to 60°N. The poles come from LOLA.

This notebook cuts the band into square tiles in the same format as `public/data/global-height` (float32 little-endian grid plus a JSON sidecar), so the globe can load sharper terrain only where the camera is looking.

Runs on a free CPU runtime in Colab or Kaggle (on Kaggle, turn on **Internet** in the notebook settings). **Runtime → Run all.**

In [ ]:
%pip -q install rasterio matplotlib requests

## 1. Settings
Change these, then run everything. Start small: one or two tiles take a minute, the full band takes much longer.

In [ ]:
SOURCE_PPD = 128         # pixels per degree to read: 128 (~237 m), 256 (~118 m) or 512 (~59 m)
READ_MODE = "download"   # "download" (fast, needs disk) or "stream" (reads over HTTP, slow, no disk)
LAT_RANGE = (-60, 60)    # degrees; SLDEM covers -60 to 60 only
LON_RANGE = (-180, 180)  # degrees east, -180 to 180 like global-height
TILE_DEG = 15            # each tile is TILE_DEG x TILE_DEG degrees (must divide 180)
TILE_PX = 512            # tile width/height in pixels. 15 deg / 512 px is about 0.03 deg (~890 m) per pixel
ASSET_NAME = "sldem"
USE_DRIVE = False        # True: save exports to Google Drive (Colab only)
DOWNLOAD_ZIP = True      # Colab: download the zip at the end

## 2. Setup
Helper functions for downloading, saving, and zipping. Nothing to change here.

In [ ]:
import json
import sys
import time
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rasterio.warp
import requests
from rasterio.enums import Resampling
from rasterio.windows import from_bounds

IN_COLAB = "google.colab" in sys.modules

# Raw downloads always stay on the runtime disk. Only finished exports go to Drive.
RAW = Path("data/raw")
OUT_ROOT = Path("data/work")
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/lunar-atlas")
EXPORT = OUT_ROOT / "exports" / ASSET_NAME
RAW.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)

MOON_RADIUS_M = 1737400
# Moon lat/lon (same sphere as the global-height export).
MOON_GEOGRAPHIC = rasterio.crs.CRS.from_proj4("+proj=longlat +R=1737400 +no_defs")


def remote_size(url):
    """Ask the server how big a file is (HTTP HEAD) without downloading it."""
    response = requests.head(url, allow_redirects=True, timeout=30)
    response.raise_for_status()
    return int(response.headers.get("Content-Length", 0))


def download(url, name=None):
    """Download once into data/raw. Re-running the cell skips finished files."""
    path = RAW / (name or Path(url).name)
    if path.exists():
        return path
    temporary_path = path.with_suffix(path.suffix + ".part")
    with requests.get(url, stream=True, timeout=(15, 120)) as response:
        response.raise_for_status()
        with temporary_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
    temporary_path.replace(path)
    return path


def save_grid(elevation, metadata, folder, name):
    """Write <name>.f32 (float32 little-endian, row-major) and <name>.json."""
    elevation = np.asarray(elevation, dtype="<f4")
    if not np.isfinite(elevation).any():
        raise ValueError(f"{name}: no valid elevations found.")
    folder.mkdir(parents=True, exist_ok=True)
    metadata = dict(metadata)
    metadata["minElevationM"] = float(np.nanmin(elevation))
    metadata["maxElevationM"] = float(np.nanmax(elevation))
    elevation.tofile(folder / f"{name}.f32")
    (folder / f"{name}.json").write_text(json.dumps(metadata, indent=2) + "\n")
    return metadata


def zip_folder(folder, zip_path):
    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as bundle:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, path.relative_to(folder.parent))
    return zip_path


def folder_size_mb(folder):
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) / 1e6

print("Colab" if IN_COLAB else "Not Colab (Kaggle or local)", "| exports go to", EXPORT.resolve())

## 3. Pick the source files
The 128 px/deg data is one global file. The 256 and 512 px/deg data are split into regional files, so only the ones touching your area are used. Sizes come from the server (HTTP HEAD).

In [ ]:
BASE = "https://imbrium.mit.edu/DATA/SLDEM2015"
SCALE = 0.5          # stored number x 0.5 = height in meters above the 1,737.4 km sphere
NODATA_DN = -32768   # "no data" marker in the source files


def source_catalog(ppd):
    """Each entry: URL and the box it covers (west, south, east, north) in its own longitudes."""
    if ppd == 128:
        name = "SLDEM2015_128_60S_60N_000_360"   # despite the name, this file runs -180 to 180
        return [{"name": name, "url": f"{BASE}/GLOBAL/JP2/{name}.JP2",
                 "label": f"{BASE}/GLOBAL/JP2/{name}_JP2.LBL", "bounds": (-180, -60, 180, 60)}]
    if ppd == 256:
        bands = {"60S_0S": (-60, 0), "0N_60N": (0, 60)}
        lon_step = 120
    elif ppd == 512:
        bands = {"60S_30S": (-60, -30), "30S_00S": (-30, 0), "00N_30N": (0, 30), "30N_60N": (30, 60)}
        lon_step = 45
    else:
        raise ValueError("SOURCE_PPD must be 128, 256 or 512")
    catalog = []
    for band, (south, north) in bands.items():
        for west in range(0, 360, lon_step):   # regional files use 0 to 360 east
            name = f"SLDEM2015_{ppd}_{band}_{west:03d}_{west + lon_step:03d}"
            catalog.append({"name": name, "url": f"{BASE}/TILES/JP2/{name}.JP2",
                            "label": f"{BASE}/TILES/JP2/{name}_JP2.LBL",
                            "bounds": (west, south, west + lon_step, north)})
    return catalog


def overlap(box, source_bounds):
    """Overlap of an output box (lon -180..180) with a source file, in the source's longitudes."""
    west, south, east, north = box
    sw, ss, se, sn = source_bounds
    if se > 180:                     # source uses 0..360: shift the box to match
        west, east = west % 360, west % 360 + (east - box[0])
    w, s, e, n = max(west, sw), max(south, ss), min(east, se), min(north, sn)
    if w >= e or s >= n:
        return None
    return (w, s, e, n), west        # also return the box's west edge in source longitudes


assert 180 % TILE_DEG == 0, "TILE_DEG must divide 180"
assert -60 <= LAT_RANGE[0] < LAT_RANGE[1] <= 60, "SLDEM covers -60 to 60 only"
assert all((v - LAT_RANGE[0]) % TILE_DEG == 0 for v in LAT_RANGE), "LAT_RANGE must be whole tiles"
assert all((v - LON_RANGE[0]) % TILE_DEG == 0 for v in LON_RANGE), "LON_RANGE must be whole tiles"
assert LAT_RANGE[0] % TILE_DEG == 0 and LON_RANGE[0] % TILE_DEG == 0, "ranges must line up with TILE_DEG"

tiles = [(west, south, west + TILE_DEG, south + TILE_DEG)
         for south in range(LAT_RANGE[0], LAT_RANGE[1], TILE_DEG)
         for west in range(LON_RANGE[0], LON_RANGE[1], TILE_DEG)]
sources = [s for s in source_catalog(SOURCE_PPD)
           if any(overlap(t, s["bounds"]) for t in tiles)]
total = 0
for s in sources:
    s["bytes"] = remote_size(s["url"])
    total += s["bytes"]
    print(f'{s["name"]}.JP2  {s["bytes"] / 1e6:,.1f} MB')
print(f"{len(sources)} source file(s), {total / 1e6:,.1f} MB to read")
print(f"{len(tiles)} output tiles, about {len(tiles) * TILE_PX * TILE_PX * 4 / 1e6:,.1f} MB of exports")

## 4. Get the data
`download` saves each file once (re-running skips it). `stream` reads only the parts it needs over the internet, which avoids the disk but is much slower.

In [ ]:
for s in sources:
    if READ_MODE == "download":
        path = download(s["url"])
        # The _AUX.XML file holds the scale and no-data value. GDAL looks for "<file>.aux.xml".
        download(s["url"].replace(".JP2", "_AUX.XML"), name=path.name + ".aux.xml")
        s["path"] = str(path)
    else:
        # vrt:// lets us tell GDAL the no-data value without the aux file.
        s["path"] = f'vrt:///vsicurl/{s["url"]}?a_nodata={NODATA_DN}'
    print("ready:", s["name"])

## 5. Cut the tiles
Each tile is read straight at the output size. `average` resampling blends the source pixels inside each output pixel, which keeps the terrain smooth instead of noisy.

In [ ]:
pixel_deg = TILE_DEG / TILE_PX
tiles_folder = EXPORT / "tiles"
index = []
start = time.time()
for number, box in enumerate(tiles, 1):
    west, south, east, north = box
    elevation = np.full((TILE_PX, TILE_PX), np.nan, dtype="<f4")
    used = []
    for s in sources:
        hit = overlap(box, s["bounds"])
        if hit is None:
            continue
        (w, so, e, n), box_west = hit
        col0, col1 = round((w - box_west) / pixel_deg), round((e - box_west) / pixel_deg)
        row0, row1 = round((north - n) / pixel_deg), round((north - so) / pixel_deg)
        with rasterio.open(s["path"]) as dataset:
            values = dataset.read(1, window=from_bounds(w, so, e, n, dataset.transform),
                                  out_shape=(row1 - row0, col1 - col0), masked=True,
                                  out_dtype="float32", resampling=Resampling.average)
            crs_wkt = dataset.crs.to_wkt()
        elevation[row0:row1, col0:col1] = (values * SCALE).filled(np.nan)
        used.append(s["url"])
    name = f"lat{south:+03d}_lon{west:+04d}"
    metadata = save_grid(elevation, {
        "source": used[0] if len(used) == 1 else used,
        "productLabel": [s["label"] for s in sources if s["url"] in used],
        "product": f"SLDEM2015 {SOURCE_PPD} px/deg (LOLA + SELENE TC)",
        "width": TILE_PX, "height": TILE_PX, "encoding": "float32-le",
        "units": "m", "order": "row-major", "nodataEncoding": "NaN",
        "referenceRadiusM": MOON_RADIUS_M,
        "globalExtentDegrees": None,
        "extentDegrees": [west, south, east, north],
        "resolutionDeg": pixel_deg,
        "approxResolutionMAtEquator": round(MOON_RADIUS_M * np.radians(pixel_deg), 1),
        "crsWkt": crs_wkt,
        "affine": [pixel_deg, 0.0, float(west), 0.0, -pixel_deg, float(north)],
        "sourceScale": SCALE, "sourceOffset": 0.0, "resampling": "average",
        "citation": "Barker et al. (2016), Icarus 273, 346-355, doi:10.1016/j.icarus.2015.07.039",
    }, tiles_folder, name)
    index.append({"name": name, "file": f"tiles/{name}.f32", "metadata": f"tiles/{name}.json",
                  "extentDegrees": [west, south, east, north],
                  "minElevationM": metadata["minElevationM"], "maxElevationM": metadata["maxElevationM"]})
    print(f"{number}/{len(tiles)} {name}  {metadata['minElevationM']:,.0f} to "
          f"{metadata['maxElevationM']:,.0f} m  ({time.time() - start:,.0f} s)")

## 6. Index and quick look
`index.json` lists every tile and its box, so the app can find which tiles to load.

In [ ]:
(EXPORT / "index.json").write_text(json.dumps({
    "product": f"SLDEM2015 {SOURCE_PPD} px/deg (LOLA + SELENE TC)",
    "productPage": "https://pgda.gsfc.nasa.gov/products/54",
    "tileDegrees": TILE_DEG, "tilePixels": TILE_PX, "resolutionDeg": pixel_deg,
    "latRange": list(LAT_RANGE), "lonRange": list(LON_RANGE),
    "encoding": "float32-le", "units": "m", "nodataEncoding": "NaN",
    "referenceRadiusM": MOON_RADIUS_M,
    "minElevationM": min(t["minElevationM"] for t in index),
    "maxElevationM": max(t["maxElevationM"] for t in index),
    "tiles": index,
}, indent=2) + "\n")

# Stitch a small preview: about 64 x 64 pixels from each tile.
step = max(1, TILE_PX // 64)
rows = []
for south in range(LAT_RANGE[1] - TILE_DEG, LAT_RANGE[0] - 1, -TILE_DEG):
    row = [np.fromfile(tiles_folder / f"lat{south:+03d}_lon{west:+04d}.f32", dtype="<f4")
           .reshape(TILE_PX, TILE_PX)[::step, ::step]
           for west in range(LON_RANGE[0], LON_RANGE[1], TILE_DEG)]
    rows.append(np.hstack(row))
preview = np.vstack(rows)
plt.figure(figsize=(12, 12 * preview.shape[0] / preview.shape[1] + 1))
plt.imshow(preview, cmap="terrain", extent=[LON_RANGE[0], LON_RANGE[1], LAT_RANGE[0], LAT_RANGE[1]])
plt.colorbar(label="Elevation (m)", shrink=0.6)
plt.xlabel("Longitude (°E)"); plt.ylabel("Latitude (°)")
plt.show()

## 7. Zip and download
Unzip into `public/data/` so you get `public/data/sldem/index.json` and `public/data/sldem/tiles/`. The full band is big, so think about committing only the tiles you use.

In [ ]:
zip_path = zip_folder(EXPORT, OUT_ROOT / f"{ASSET_NAME}.zip")
print(f"Exports: {folder_size_mb(EXPORT):.1f} MB in {EXPORT}")
print(f"Zip: {zip_path.stat().st_size / 1e6:.1f} MB at {zip_path}")
if DOWNLOAD_ZIP and IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Download the zip from the file panel (Kaggle: Output tab).")